# TV2 — 01. Khám phá dữ liệu và sự kiện oddball
Giai đoạn 1: kiểm tra số người, tên tác vụ, participants.tsv.

In [ ]:
import os, sys, glob, re, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
from eegoddball import config  # cần `pip install -e .`, cùng kiểu import với create_v0_data.py
print(config.DATA_DIR)

## 1.2 Đếm file oddball .vhdr và tìm người thiếu

In [ ]:
vhdr = sorted(glob.glob(os.path.join(config.DATA_DIR, "sub-*", "eeg", "*task-oddball*_eeg.vhdr")))
found = [re.search(r"sub-(\d+)", os.path.basename(f)).group(1) for f in vhdr]
missing = sorted(set(config.SUBJECTS) - set(found))
print(f"Số file oddball .vhdr: {len(vhdr)} / {len(config.SUBJECTS)}")
print("Người thiếu:", missing or "không có")

## 1.3 Tên tác vụ thật trong tên file

In [ ]:
all_vhdr = glob.glob(os.path.join(config.DATA_DIR, "sub-*", "eeg", "*_eeg.vhdr"))
tasks = pd.Series([re.search(r"task-([^_]+)", os.path.basename(f)).group(1) for f in all_vhdr])
print("Tác vụ có file .vhdr:\n", tasks.value_counts())

## 1.4 participants.tsv

In [ ]:
participants = pd.read_csv(os.path.join(config.DATA_DIR, "participants.tsv"), sep="\t")
print(participants.shape)
participants.head()

# Giai đoạn 2: Sự kiện của sub-01
## 2.1 Đọc events.tsv: cột, 20 dòng đầu, value_counts của nhãn

In [ ]:
def read_events(sub):
    path = os.path.join(config.DATA_DIR, f"sub-{sub}", "eeg", f"sub-{sub}_task-oddball_events.tsv")
    ev = pd.read_csv(path, sep="\t")
    # chuẩn hóa khoảng trắng "S  5" -> "S 5" giống create_v0_data.read_stimulus_metadata
    ev["code"] = ev["event_type"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    return ev

ev = read_events("01")
print("Cột:", ev.columns.tolist())
print("Mã event_type gốc (repr để thấy dấu cách):", [repr(c) for c in sorted(ev["event_type"].unique())])
print(ev["event_type"].value_counts().sort_index(), "\n")
print(pd.crosstab(ev["event_type"], ev["trial_type"].fillna("NaN")), "\n")
ev.head(20)

## 2.2 Nhãn nào là standard / target / distractor / phản hồi
Đối chiếu mô tả trong `task-oddball_events.json` với số lượng thật.

In [ ]:
with open(os.path.join(config.DATA_DIR, "task-oddball_events.json"), encoding="utf-8") as f:
    levels = json.load(f)["event_type"]["Levels"]
print(pd.DataFrame({"mô tả (json)": pd.Series(levels), "số lượng sub-01": ev["event_type"].value_counts()}))

CODE_TO_CLASS = config.STIMULUS_CODES  # {"S 5": "standard", "S 6": "target", "S 7": "distractor"}
RESPONSE_CODE = "S 1"  # theo task-oddball_events.json, sau khi chuẩn hóa khoảng trắng
assert set(CODE_TO_CLASS.values()) == set(config.EVENT_ID)
print({code: (cls, config.EVENT_ID[cls]) for code, cls in CODE_TO_CLASS.items()})

## 2.3 Số lượng và tỷ lệ mỗi loại kích thích

In [ ]:
stim = ev[ev["code"].isin(CODE_TO_CLASS)].sort_values("onset").reset_index(drop=True)
stim["label"] = stim["code"].map(CODE_TO_CLASS)
counts = stim["label"].value_counts()
print(pd.DataFrame({"n": counts, "tỷ lệ %": (100 * counts / len(stim)).round(1)}))
print("Tổng kích thích:", len(stim), "| standard : target =", round(counts["standard"] / counts["target"], 2))

## 2.4 Phản hồi sau mỗi kích thích và độ trễ (RT)
Mỗi phản hồi được gắn vào kích thích gần nhất **trước** nó, tức cửa sổ phản hồi của một kích thích kéo dài từ onset của nó đến onset kích thích kế tiếp.

In [ ]:
def pair_responses(ev):
    stim = ev[ev["code"].isin(CODE_TO_CLASS)].sort_values("onset").reset_index(drop=True)
    stim["label"] = stim["code"].map(CODE_TO_CLASS)
    resp_on = np.sort(ev.loc[ev["code"] == RESPONSE_CODE, "onset"].to_numpy())
    # vị trí kích thích gần nhất đứng trước mỗi phản hồi
    idx = np.searchsorted(stim["onset"].to_numpy(), resp_on, side="right") - 1
    assert (idx >= 0).all(), "có phản hồi trước kích thích đầu tiên"
    rt = pd.Series(resp_on - stim["onset"].to_numpy()[idx], index=idx)
    stim["n_resp"] = rt.groupby(level=0).size().reindex(stim.index, fill_value=0)
    stim["rt"] = rt.groupby(level=0).min().reindex(stim.index)  # RT của phản hồi đầu tiên
    return stim

stim = pair_responses(ev)
summary = stim.groupby("label").agg(
    n=("onset", "size"), co_bam=("n_resp", lambda s: (s > 0).sum()),
    rt_min=("rt", "min"), rt_median=("rt", "median"), rt_max=("rt", "max"))
summary["ty_le_bam_%"] = (100 * summary["co_bam"] / summary["n"]).round(1)
print("Khoảng cách giữa 2 kích thích (min):", np.diff(stim["onset"]).min().round(3), "s")
print("Kích thích có >1 phản hồi:", (stim["n_resp"] > 1).sum())
print("Target KHÔNG có phản hồi:", ((stim["label"] == "target") & (stim["n_resp"] == 0)).sum())
summary.round(3)

## 2.5 Nhất quán nhãn và hành vi qua 3 người đầu

In [ ]:
rows = []
for sub in config.SAMPLE_SUBJECTS:
    e = read_events(sub)
    s = pair_responses(e)
    pressed = (s["n_resp"] > 0).groupby(s["label"]).mean().mul(100).round(1)
    rows.append({"sub": sub, "mã event_type": " | ".join(sorted(e["event_type"].unique())),
                 **s["label"].value_counts().to_dict(),
                 **pressed.add_prefix("bấm % ").to_dict(),
                 "RT target median": s.loc[s["label"] == "target", "rt"].median().round(3),
                 "kích thích >1 phản hồi": int((s["n_resp"] > 1).sum())})
pd.DataFrame(rows).set_index("sub").T

## 2.6 Số thử nghiệm còn lại theo quy tắc "làm đúng" (đã chốt)
target đúng = có bấm; standard/distractor đúng = không bấm (trong cửa sổ đến kích thích kế tiếp). Không đặt ngưỡng RT; sub-02 giữ đúng hợp đồng.

In [ ]:
kept = {}
for sub in config.SAMPLE_SUBJECTS:
    s = pair_responses(read_events(sub))
    s["correct"] = (s["label"] == "target") == (s["n_resp"] > 0)
    kept[sub] = s.groupby("label")["correct"].agg(["sum", "size"]).astype(str).agg("/".join, axis=1)
pd.DataFrame(kept)